# 209. LLM 输出 PII：扫描、脱敏与 Egress Policy Gate 怎样实现？

> **面试问题：怎样在模型输出/工具外发前定位 PII span、按风险掩码或阻断、处理未知格式，并以最小化审计和 precision/recall 验收？**

## 先给结论

这里的关键不是调用一个安全/训练/推理框架，而是定义输入、状态、不变量、失败分支和独立的判断 oracle。下方仅以受控小数据验证机制；真实服务仍需替换模型、密钥管理、访问控制、审计、红队评测和线上 SLO。

## 一手资料

- [Dolma](https://arxiv.org/abs/2402.00159)
- [NIST AI RMF](https://www.nist.gov/itl/ai-risk-management-framework)
- [AgentDojo](https://arxiv.org/abs/2406.13352)

In [ ]:
notebook_contract = {"mode": "controlled-demo", "oracle": "assertions", "production": "security-and-versioning-required"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "controlled-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "versioning" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：输出 PII gate 是模型后的独立安全边界

输入去 PII 不代表模型永远不会复述敏感内容。输出侧需要在呈现/工具 egress 前扫描、分类、按策略掩码或阻断，并保存最小化审计摘要；正则只是高精度 baseline，不能覆盖所有身份信息。


In [ ]:
import re  # 执行本行的状态、计算或校验逻辑。
text = "请联系 alice@example.com，电话 13812345678，订单号 A-42。"  # 执行本行的状态、计算或校验逻辑。
patterns = {"email": re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}"), "phone": re.compile(r"1[3-9]\d{9}")}  # 执行本行的状态、计算或校验逻辑。
assert "alice@example.com" in text  # 执行本行的状态、计算或校验逻辑。
assert "13812345678" in text  # 执行本行的状态、计算或校验逻辑。
assert set(patterns) == {"email", "phone"}  # 执行本行的状态、计算或校验逻辑。


## 2. 扫描：finding 必须含类型和 span，而非只返回布尔值

span 允许后续确定性 redaction、重叠处理和审计计数。生产系统还应有国家/语言规则、上下文验证、NER/分类器、人审通道，以及对误报/漏报的持续评估。


In [ ]:
def scan(text, patterns):  # 执行本行的状态、计算或校验逻辑。
    findings = []  # 执行本行的状态、计算或校验逻辑。
    for kind, pattern in patterns.items():  # 执行本行的状态、计算或校验逻辑。
        for match in pattern.finditer(text):  # 执行本行的状态、计算或校验逻辑。
            findings.append({"kind": kind, "start": match.start(), "end": match.end()})  # 执行本行的状态、计算或校验逻辑。
    return sorted(findings, key=lambda item: item["start"])  # 执行本行的状态、计算或校验逻辑。
findings = scan(text, patterns)  # 执行本行的状态、计算或校验逻辑。
assert [item["kind"] for item in findings] == ["email", "phone"]  # 执行本行的状态、计算或校验逻辑。
assert text[findings[0]["start"]:findings[0]["end"]] == "alice@example.com"  # 执行本行的状态、计算或校验逻辑。
assert len(findings) == 2  # 执行本行的状态、计算或校验逻辑。


## 3. 策略：按数据类型和动作风险选择 mask 或 block

不是所有命中都应一律删除：某些客服工作流可显示受权限保护的末四位，外发工具则应阻断。下面使用最简 policy；真实决策还要考虑数据主体、用途、租户、地区和用户批准。


In [ ]:
policy = {"email": "mask", "phone": "block"}  # 执行本行的状态、计算或校验逻辑。
def policy_decision(findings, policy):  # 执行本行的状态、计算或校验逻辑。
    actions = [policy[item["kind"]] for item in findings]  # 执行本行的状态、计算或校验逻辑。
    return "block" if "block" in actions else "mask" if "mask" in actions else "allow"  # 执行本行的状态、计算或校验逻辑。
assert policy_decision(findings, policy) == "block"  # 执行本行的状态、计算或校验逻辑。
assert policy_decision(findings[:1], policy) == "mask"  # 执行本行的状态、计算或校验逻辑。
assert policy_decision([], policy) == "allow"  # 执行本行的状态、计算或校验逻辑。


## 4. 掩码：从右向左替换，避免 span 位移

多个 finding 的替换会改变字符串长度，故要从后向前处理。mask 后不应将原敏感值写入返回对象或普通日志；需要明文的受控系统应走专用密钥/权限服务。


In [ ]:
def redact(text, findings):  # 执行本行的状态、计算或校验逻辑。
    output = text  # 执行本行的状态、计算或校验逻辑。
    for item in reversed(findings):  # 执行本行的状态、计算或校验逻辑。
        placeholder = f"<{item['kind'].upper()}>"  # 执行本行的状态、计算或校验逻辑。
        output = output[:item["start"]] + placeholder + output[item["end"]:]  # 执行本行的状态、计算或校验逻辑。
    return output  # 执行本行的状态、计算或校验逻辑。
redacted = redact(text, findings)  # 执行本行的状态、计算或校验逻辑。
assert "alice@example.com" not in redacted  # 执行本行的状态、计算或校验逻辑。
assert "13812345678" not in redacted  # 执行本行的状态、计算或校验逻辑。
assert "<EMAIL>" in redacted and "<PHONE>" in redacted  # 执行本行的状态、计算或校验逻辑。


## 5. 执行门禁：block 时不得把原输出发送到外部工具

安全 gate 应位于最终 egress 前，独立于模型的自我声明。返回给上层的是决策、掩码文本和计数；在 block 情况下，可以请求人工审批或让模型在不含 PII 的前提下重写。


In [ ]:
def guard_output(text, patterns, policy):  # 执行本行的状态、计算或校验逻辑。
    findings = scan(text, patterns)  # 执行本行的状态、计算或校验逻辑。
    decision = policy_decision(findings, policy)  # 执行本行的状态、计算或校验逻辑。
    return {"decision": decision, "safe_text": redact(text, findings), "counts": {kind: sum(item["kind"] == kind for item in findings) for kind in patterns}}  # 执行本行的状态、计算或校验逻辑。
guarded = guard_output(text, patterns, policy)  # 执行本行的状态、计算或校验逻辑。
assert guarded["decision"] == "block"  # 执行本行的状态、计算或校验逻辑。
assert guarded["counts"] == {"email": 1, "phone": 1}  # 执行本行的状态、计算或校验逻辑。
assert "alice@example.com" not in guarded["safe_text"]  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：未知格式和正则误报必须可升级

正则只是一层检测，不能声称零漏报。对于未知模式、重叠 span 或高风险领域，系统应当标记 review_required 或强制更严格的检测器；不要以“未命中”自动证明内容安全。


In [ ]:
def conservative_decision(findings, detector_coverage, high_risk):  # 执行本行的状态、计算或校验逻辑。
    if high_risk and not detector_coverage:  # 执行本行的状态、计算或校验逻辑。
        return "review_required"  # 执行本行的状态、计算或校验逻辑。
    return "allow" if not findings else "inspect_findings"  # 执行本行的状态、计算或校验逻辑。
assert conservative_decision([], False, True) == "review_required"  # 执行本行的状态、计算或校验逻辑。
assert conservative_decision([], True, False) == "allow"  # 执行本行的状态、计算或校验逻辑。
assert conservative_decision(findings, True, False) == "inspect_findings"  # 执行本行的状态、计算或校验逻辑。


## 7. 评测：precision、recall、业务阻断率都要单独报

只优化命中数会导致大规模误报或漏报。评测集应按 PII 类型、语言、格式、攻击式拼写和业务上下文切片，分别报告 span precision/recall、用户打断率与人工复核延迟。


In [ ]:
import math  # 执行本行的状态、计算或校验逻辑。
def precision_recall(true_positive, false_positive, false_negative):  # 执行本行的状态、计算或校验逻辑。
    precision = true_positive / max(1, true_positive + false_positive)  # 执行本行的状态、计算或校验逻辑。
    recall = true_positive / max(1, true_positive + false_negative)  # 执行本行的状态、计算或校验逻辑。
    return precision, recall  # 执行本行的状态、计算或校验逻辑。
precision, recall = precision_recall(8, 2, 1)  # 执行本行的状态、计算或校验逻辑。
assert precision == 0.8  # 执行本行的状态、计算或校验逻辑。
assert math.isclose(recall, 8 / 9)  # 执行本行的状态、计算或校验逻辑。
assert recall > precision  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：审计保存计数与版本，避免再次泄露原文

日志中应保存 policy/detector 版本、类别计数、决策和请求匿名 id，而不是原始 PII。安全事件的原文如必须保留，应转入受权限/保留期控制的专用系统。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
audit = {"request": "anon-r-1", "policy": "pii-v1", "detectors": sorted(patterns), "decision": guarded["decision"], "counts": guarded["counts"]}  # 执行本行的状态、计算或校验逻辑。
fingerprint = hashlib.sha256(json.dumps(audit, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert "alice@example.com" not in json.dumps(audit)  # 执行本行的状态、计算或校验逻辑。
assert audit["decision"] == "block"  # 执行本行的状态、计算或校验逻辑。
assert len(fingerprint) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

完整回答应覆盖目标、显式数据结构、核心规则、边界失败、评测指标和版本制品。受控断言只验证实现不变量，不能被解读为真实模型质量、攻击鲁棒性、隐私合规或线上成本结论。
